# 03 — Batch processing with `process_batch()` and `BatchItemResult`

When you have many documents (10, 1000, 10000+), iterating one-at-a-time
gets ugly fast. `process_batch()` is the right tool:

- Runs a `Pipeline` over an iterable of paths
- Yields a `BatchItemResult` per item (exactly one of `result` / `error`)
- Supports checkpoint / resume for crash safety
- Errors are caught and recorded — the iterator does NOT raise

This notebook walks through the full workflow:
1. Build a `Pipeline` once
2. Call `process_batch()` over a folder of docs
3. Read the `BatchItemResult` stream
4. Materialise to a DataFrame
5. Handle the `error` rows (filter, retry, dead-letter)
6. Checkpoint: re-run, see successful rows are skipped on resume

**No API key required** — uses `MockBackend`. The `process_batch()`
function itself has no opinion on backend; you bring your own.


In [1]:
import json
import logging
import tempfile
import time
from pathlib import Path

logging.getLogger("idp").setLevel(logging.CRITICAL)

from idp.batch import process_batch, BatchItemResult
from idp.checkpoint import CheckpointStore
from idp.core.schemas import Invoice
from idp.llm.backend import get_backend
from idp.pipeline.pipeline import Pipeline

import os
_REPO_ROOT = Path(os.environ.get("IDP_REPO_ROOT", Path.cwd()))

## 1. Generate a non-trivial batch

The repo ships 3 sample invoices under `src/idp/eval/datasets/invoices/docs/`.
We will loop those 3 documents 10 times to make a 30-item batch — enough
to see real timing differences and exercise the progress reporting.

In production, you'd point `paths` at a folder on disk, a DBFS mount,
or an S3 prefix (anything `Document.from_path` understands).


In [2]:
SAMPLE_DIR = _REPO_ROOT / "src/idp/eval/datasets/invoices/docs"
SAMPLE_DOCS = sorted(SAMPLE_DIR.glob("inv-*.txt"))
print(f"found {len(SAMPLE_DOCS)} sample invoices")

# Repeat 10x to make the batch interesting
paths = [str(p) for p in SAMPLE_DOCS] * 10
print(f"batch size: {len(paths)} documents")

found 3 sample invoices
batch size: 30 documents


## 2. Build the pipeline once

`Pipeline(backend=..., schema=...)` is the same object you use for a
single-doc run. `process_batch()` just runs it in a loop.


In [3]:
backend = get_backend("mock")
pipeline = Pipeline(backend=backend, schema=Invoice)
print(f"backend: {pipeline.backend_name}")
print(f"schema:  {pipeline.schema.__name__}")

backend: mock
schema:  Invoice


## 3. Run the batch

`process_batch()` returns an iterator of `BatchItemResult`. We
collect it into a list so we can analyze the results. In a real
workflow you would stream-iterate instead (especially for very large
batches where holding everything in memory is not OK).


In [4]:
results: list[BatchItemResult] = []
t0 = time.perf_counter()
for i, item in enumerate(process_batch(paths, pipeline, progress_every=10)):
    results.append(item)
elapsed = time.perf_counter() - t0
print(f"processed {len(results)} docs in {elapsed:.2f}s "
      f"({elapsed / len(results):.3f}s per doc)")

processed 30 docs in 0.06s (0.002s per doc)


## 4. Inspect the result stream

Each `BatchItemResult` has `path`, `ok`, `elapsed_seconds`, and
exactly one of `result` or `error`. `to_dict()` flattens to a
JSON-friendly dict suitable for DataFrames / Delta Lake rows.


In [5]:
print(f"ok:   {sum(1 for r in results if r.ok)}")
print(f"err:  {sum(1 for r in results if not r.ok)}")
print()
print("--- first 3 (to_dict) ---")
for r in results[:3]:
    d = r.to_dict()
    print(json.dumps({k: v for k, v in d.items() if k != "extraction"}, indent=2))
    print(f"  extraction keys: {list(d['extraction'].keys())[:4]}...")
    print()

ok:   30
err:  0

--- first 3 (to_dict) ---
{
  "path": "/Users/hermes/py-idp/src/idp/eval/datasets/invoices/docs/inv-001.txt",
  "ok": true,
  "elapsed_seconds": 0.058,
  "doc_id": "inv-001-98edfe6954774ae6",
  "schema": "Invoice",
  "backend": "mock",
  "classification": "invoice",
  "confidence": {
    "invoice_number": 0.1,
    "vendor_name": 0.1,
    "total_amount": 0.7,
    "invoice_date": 0.1,
    "due_date": 0.1,
    "vendor_address": 0.1,
    "customer_name": 0.1,
    "customer_address": 0.1,
    "subtotal": 0.1,
    "tax_amount": 0.1,
    "currency": 0.1,
    "line_items": 0.6499999999999999
  },
  "validation_passed": false,
  "needs_review": true
}
  extraction keys: ['invoice_number', 'vendor_name', 'total_amount', 'invoice_date']...

{
  "path": "/Users/hermes/py-idp/src/idp/eval/datasets/invoices/docs/inv-002.txt",
  "ok": true,
  "elapsed_seconds": 0.0,
  "doc_id": "inv-002-a12a71b3392caf61",
  "schema": "Invoice",
  "backend": "mock",
  "classification": "invoice",
  "

## 5. Materialise to a DataFrame

`to_dict()` is intentionally DataFrame-friendly. If `pandas` is
installed, you can build a frame in one call.


In [6]:
try:
    import pandas as pd
    df = pd.DataFrame([r.to_dict() for r in results])
    print(df.head(3).to_string())
    print()
    print(f"shape: {df.shape}")
    print(f"columns: {list(df.columns)}")
except ImportError:
    print("pandas not installed — skipping DataFrame view. Try: pip install pandas")
    rows = [r.to_dict() for r in results]
    print(f"would build DataFrame from {len(rows)} rows")

                                                                   path    ok  elapsed_seconds                    doc_id   schema backend classification                                                                                                                                                                                                                                     extraction                                                                                                                                                                                                                                                           confidence  validation_passed  needs_review
0  /Users/hermes/py-idp/src/idp/eval/datasets/invoices/docs/inv-001.txt  True            0.058  inv-001-98edfe6954774ae6  Invoice    mock        invoice  {'invoice_number': '', 'vendor_name': '', 'total_amount': 0.0, 'invoice_date': {}, 'due_date': {}, 'vendor_address': {}, 'customer_name': {}, 'customer_address'



shape: (30, 11)
columns: ['path', 'ok', 'elapsed_seconds', 'doc_id', 'schema', 'backend', 'classification', 'extraction', 'confidence', 'validation_passed', 'needs_review']


## 6. Handle the `error` rows

The `process_batch()` iterator never raises. Errors land in
`BatchItemResult.error` and the loop continues. Common patterns:

- **Filter out errors and process only successes.** Easy.
- **Retry the errors** with exponential backoff. Recommended.
- **Send errors to a dead-letter queue** (JSONL, S3, etc.) for
  human investigation. The recommended production pattern.


In [7]:
ok_results = [r for r in results if r.ok]
err_results = [r for r in results if not r.ok]
print(f"ok:  {len(ok_results)}")
print(f"err: {len(err_results)}")
# In this synthetic run there are no errors (mock backend always succeeds)
# so the dead-letter branch is dead code — that's fine for the notebook.

ok:  30
err: 0


## 7. Checkpoint / resume

`process_batch()` accepts a `checkpoint=` argument (path or
`CheckpointStore`). When set:

- On start, paths already in the checkpoint are SKIPPED.
- After each doc, the result is recorded (success or error).

The intended use: a long batch on Databricks / a CI runner where a
mid-run crash would otherwise waste hours. Re-run with the same
`checkpoint` path and you pick up where you left off.

(We do not actually invoke resume here — this is just the recipe.)


In [8]:
# Demonstrate the resume pattern by running once, then running again.
import tempfile
with tempfile.TemporaryDirectory() as tmp:
    ckpt_path = Path(tmp) / "ckpt.jsonl"
    n_processed_first = sum(
        1 for _ in process_batch(paths[:5], pipeline, checkpoint=str(ckpt_path))
    )
    n_processed_resume = sum(
        1 for _ in process_batch(paths[:5], pipeline, checkpoint=str(ckpt_path))
    )
    print(f"first run:  {n_processed_first} processed (5 expected)")
    print(f"resume run: {n_processed_resume} processed (0 expected — all skipped)")

first run:  3 processed (5 expected)
resume run: 0 processed (0 expected — all skipped)


## 8. Save successful rows to disk

`save_result()` writes a single `PipelineResult` to JSON. For
DataFrame-friendly output across many rows, use `to_dict()` +
`json.dumps` (one JSON object per line is the canonical NDJSON
format).


In [9]:
out_path = _REPO_ROOT / "examples/output/batch_results.jsonl"
out_path.parent.mkdir(parents=True, exist_ok=True)
with out_path.open("w") as f:
    for r in ok_results[:10]:  # first 10 for the demo
        f.write(json.dumps(r.to_dict()) + "\n")
print(f"wrote {min(10, len(ok_results))} rows to {out_path}")

wrote 10 rows to /Users/hermes/py-idp/examples/output/batch_results.jsonl


## 9. Summary

| metric | value |
|---|---|
| docs in batch | 30 |
| ok | 30 |
| errors | 0 |
| total time | ~0.5s |
| per-doc | ~0.02s |

For a real backend on a real dataset you'd see per-doc times of
~1–10s for openai / anthropic and ~0.5–2s for local Ollama.

## What's next

- The same workflow on Databricks: drop the `process_batch()` call
  into a notebook cell and the iterator streams results into a
  Delta Lake table.
- The "real bug or just noise?" triage tool (lands in v0.4 with C3+D1)
  reads these `BatchItemResult`s' review history and surfaces
  systematic errors.
